# Cosmos DB Mirrored Data Transformation

This notebook processes Cosmos DB data that has been mirrored into Microsoft Fabric
as Delta tables. It:

1. Reads mirrored `dbo_conversations` and `dbo_interactions` tables
2. Flattens nested JSON structures using explicit schemas
3. Extracts agent performance metrics (response time, tokens, satisfaction)
4. Sessionizes interactions with window functions
5. Writes cleansed Silver tables and an aggregated Gold analytics table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DoubleType, TimestampType, DateType, DecimalType, ArrayType, BooleanType
)

## Read Mirrored Tables

Cosmos DB mirroring in Fabric exposes containers as Delta tables following the
`dbo_<container>` naming convention. We read both `conversations` and `interactions`.

In [ ]:
raw_conversations = spark.read.format("delta").load("Tables/dbo_conversations")
raw_interactions = spark.read.format("delta").load("Tables/dbo_interactions")

print(f"Raw conversations: {raw_conversations.count()} rows")
print(f"Raw interactions:  {raw_interactions.count()} rows")

## Flatten Nested JSON & Extract Fields

Cosmos DB documents contain nested structures serialized as JSON strings.
We define explicit `StructType` schemas and use `from_json` to parse them,
then extract agent performance metrics:

- `response_time_ms` — time from request to first response byte
- `token_usage.prompt_tokens` / `completion_tokens` — LLM token counts
- `satisfaction_score` — end-user rating (1-5 scale)

In [ ]:
metadata_schema = StructType([
    StructField("source", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("language", StringType(), True),
    StructField("client_version", StringType(), True),
    StructField("tags", ArrayType(StringType()), True)
])

participant_schema = ArrayType(StructType([
    StructField("participant_id", StringType(), True),
    StructField("role", StringType(), True),
    StructField("display_name", StringType(), True)
]))

silver_conversations = (
    raw_conversations
    .withColumn("metadata_parsed", F.from_json(F.col("metadata"), metadata_schema))
    .withColumn("participants_parsed", F.from_json(F.col("participants"), participant_schema))
    .select(
        F.col("id").alias("conversation_id").cast(StringType()),
        F.col("session_id").cast(StringType()),
        F.col("user_id").cast(StringType()),
        F.col("agent_id").cast(StringType()),
        F.col("created_at").cast(TimestampType()).alias("conversation_started_at"),
        F.col("updated_at").cast(TimestampType()).alias("conversation_updated_at"),
        F.col("status").cast(StringType()).alias("conversation_status"),
        F.col("metadata_parsed.source").alias("source_system"),
        F.col("metadata_parsed.channel").alias("channel"),
        F.col("metadata_parsed.language").alias("language"),
        F.col("metadata_parsed.client_version").alias("client_version"),
        F.col("metadata_parsed.tags").alias("tags"),
        F.size(F.col("participants_parsed")).alias("participant_count"),
        F.col("satisfaction_score").cast(DoubleType())
    )
    .withColumn("conversation_date", F.col("conversation_started_at").cast(DateType()))
)

print(f"Silver conversations prepared — {silver_conversations.count()} rows")

In [ ]:
token_usage_schema = StructType([
    StructField("prompt_tokens", IntegerType(), True),
    StructField("completion_tokens", IntegerType(), True),
    StructField("total_tokens", IntegerType(), True)
])

performance_schema = StructType([
    StructField("response_time_ms", LongType(), True),
    StructField("processing_time_ms", LongType(), True),
    StructField("model_name", StringType(), True),
    StructField("model_version", StringType(), True)
])

silver_interactions = (
    raw_interactions
    .withColumn("token_usage_parsed", F.from_json(F.col("token_usage"), token_usage_schema))
    .withColumn("performance_parsed", F.from_json(F.col("performance"), performance_schema))
    .select(
        F.col("id").alias("interaction_id").cast(StringType()),
        F.col("conversation_id").cast(StringType()),
        F.col("session_id").cast(StringType()),
        F.col("agent_id").cast(StringType()),
        F.col("interaction_type").cast(StringType()),
        F.col("timestamp").cast(TimestampType()).alias("interaction_timestamp"),
        F.col("user_message").cast(StringType()),
        F.col("agent_response").cast(StringType()),
        F.col("performance_parsed.response_time_ms").cast(LongType()).alias("response_time_ms"),
        F.col("performance_parsed.processing_time_ms").cast(LongType()).alias("processing_time_ms"),
        F.col("performance_parsed.model_name").alias("model_name"),
        F.col("performance_parsed.model_version").alias("model_version"),
        F.col("token_usage_parsed.prompt_tokens").alias("prompt_tokens"),
        F.col("token_usage_parsed.completion_tokens").alias("completion_tokens"),
        F.col("token_usage_parsed.total_tokens").alias("total_tokens"),
        F.col("status").cast(StringType()).alias("interaction_status")
    )
    .withColumn("interaction_date", F.col("interaction_timestamp").cast(DateType()))
)

print(f"Silver interactions prepared — {silver_interactions.count()} rows")

## Sessionize Interactions

Group interactions into sessions and compute session-level boundaries using `lag` and
`lead` window functions. This enriches each interaction row with its ordinal position
within the session and flags session start/end boundaries.

In [ ]:
session_window = (
    Window
    .partitionBy("session_id")
    .orderBy("interaction_timestamp")
)

silver_interactions_sessionized = (
    silver_interactions
    .withColumn("prev_interaction_ts", F.lag("interaction_timestamp").over(session_window))
    .withColumn("next_interaction_ts", F.lead("interaction_timestamp").over(session_window))
    .withColumn("interaction_order", F.row_number().over(session_window))
    .withColumn("is_session_start", F.col("prev_interaction_ts").isNull())
    .withColumn("is_session_end", F.col("next_interaction_ts").isNull())
    .withColumn("time_since_prev_ms",
        F.when(
            F.col("prev_interaction_ts").isNotNull(),
            (F.unix_timestamp("interaction_timestamp") - F.unix_timestamp("prev_interaction_ts")) * 1000
        ).otherwise(F.lit(0))
    )
    .drop("prev_interaction_ts", "next_interaction_ts")
)

print(f"Sessionized interactions — {silver_interactions_sessionized.count()} rows")

## Write Silver Tables

Persist the cleansed conversation and interaction data as Delta tables for
downstream Gold aggregation and ad-hoc analysis.

In [ ]:
(
    silver_conversations
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/silver_conversations")
)
print("silver_conversations written")

(
    silver_interactions_sessionized
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/silver_interactions")
)
print("silver_interactions written")

## Gold Agent Analytics

Aggregate interaction data into a daily agent performance summary:
- Average response time and processing time
- Average satisfaction score (joined from conversations)
- Total prompt, completion, and combined token usage
- Conversation and interaction counts
- Topic classification using keyword matching

In [ ]:
interactions_with_satisfaction = (
    silver_interactions_sessionized
    .join(
        silver_conversations.select("conversation_id", "satisfaction_score"),
        on="conversation_id",
        how="left"
    )
)

interactions_with_topic = (
    interactions_with_satisfaction
    .withColumn("topic_classification",
        F.when(F.lower(F.col("user_message")).contains("error") | F.lower(F.col("user_message")).contains("fail") | F.lower(F.col("user_message")).contains("broken"), "troubleshooting")
         .when(F.lower(F.col("user_message")).contains("how") | F.lower(F.col("user_message")).contains("help") | F.lower(F.col("user_message")).contains("guide"), "how-to")
         .when(F.lower(F.col("user_message")).contains("cost") | F.lower(F.col("user_message")).contains("price") | F.lower(F.col("user_message")).contains("billing"), "billing")
         .when(F.lower(F.col("user_message")).contains("create") | F.lower(F.col("user_message")).contains("deploy") | F.lower(F.col("user_message")).contains("provision"), "provisioning")
         .when(F.lower(F.col("user_message")).contains("permission") | F.lower(F.col("user_message")).contains("access") | F.lower(F.col("user_message")).contains("role"), "access-management")
         .otherwise("general")
    )
)

session_stats = (
    interactions_with_topic
    .groupBy("session_id")
    .agg(
        F.count("*").alias("messages_per_session"),
        F.min("interaction_timestamp").alias("session_start"),
        F.max("interaction_timestamp").alias("session_end")
    )
    .withColumn("session_duration_seconds",
        (F.unix_timestamp("session_end") - F.unix_timestamp("session_start")).cast(LongType())
    )
)

gold_agent_analytics = (
    interactions_with_topic
    .groupBy("agent_id", "interaction_date", "topic_classification")
    .agg(
        F.avg("response_time_ms").cast(DecimalType(12, 2)).alias("avg_response_time_ms"),
        F.avg("processing_time_ms").cast(DecimalType(12, 2)).alias("avg_processing_time_ms"),
        F.avg("satisfaction_score").cast(DecimalType(5, 2)).alias("avg_satisfaction_score"),
        F.sum("prompt_tokens").alias("total_prompt_tokens"),
        F.sum("completion_tokens").alias("total_completion_tokens"),
        F.sum("total_tokens").alias("total_tokens"),
        F.countDistinct("conversation_id").alias("conversation_count"),
        F.count("*").alias("interaction_count"),
        F.countDistinct("session_id").alias("session_count")
    )
    .withColumn("interaction_date", F.col("interaction_date").cast(DateType()))
)

(
    gold_agent_analytics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/gold_agent_analytics")
)

print(f"gold_agent_analytics written — {gold_agent_analytics.count()} rows")
print("\nCosmos DB mirrored data transformation complete.")